# **Saisonnalité : Précipitations**
# Présence de la saisonnalité, durée et début de la saison pluvieuse

Ce rapport génère des visualisations pour l'analyse de la saisonnalité sur base des précipitations, au niveau administratif ADM2.

Il se base sur les valeurs des paramètres utilisées dans le notebook de calcul de la saisonnalité sur base des précipitations, faisant partie du même pipeline. Il s'agit des choix suivants
* Le nombre minimum de mois composant le bloc saisonnier
* Le nombre maximum de mois composant le bloc saisonnier
* La proportion minimale, _par année_, de précipitations qui doivent avoir lieu durant le bloc saisonnier, pour conclure que l'unité administrative est saisonnière
* La proportion minimale d'années pendant lesquelles il faut observer la tendance saisonnière pour conclure que l'unité administrative est saisonnière
* Le mode de calcul du dénominateur du pourcentage de précipitations:
    - [le "forward-facing sliding window" de 12 mois proposée par l'OMS](https://www.who.int/publications/i/item/9789240115712)
    - le total des précipitations de l'année calendaire correspondant au numérateur (le pourcentage de précipitations annuelles)

Le rapport produit notamment les graphiques suivants :
- Carte de classification de la saisonnalité par unité administrative (saisonnière ou non saisonnière)
- Carte du mois de début de saison pluvieuse
- Carte de la durée de la saison pluvieuse
- Carte de la proportion des précipitations qui ont lieu durant le bloc saisonnier

## 1. Configuration

In [ ]:
rm(list = ls())

In [1]:
# Global settings and environments
options(scipen=999)

Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

# Paths
ROOT_PATH <- '~/workspace'
PIPELINE_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_seasonality_rainfall')
CODE_PATH <- file.path(ROOT_PATH, 'code')
UTILS_PATH <- file.path(PIPELINE_PATH, 'utils')

In [ ]:
# Load utils and bootstrap context
source(file.path(CODE_PATH, "snt_utils.r"))
source(file.path(CODE_PATH, "snt_report.r"))
source(file.path(CODE_PATH, "snt_palettes.r"))
source(file.path(UTILS_PATH, "snt_seasonality_rainfall.r"))

pipeline_msg("Importation des fonctions, librairies et dépendances en cours.")

setup_ctx <- bootstrap_seasonality_rainfall_context(
  root_path = ROOT_PATH,
  required_packages = c(
    "jsonlite", "data.table", "ggplot2", "ggridges", "dplyr", "arrow", "glue",
    "sf", "RColorBrewer", "forcats", "httr", "reticulate", "IRdisplay"
  )
)

CONFIG_PATH <- setup_ctx$CONFIG_PATH
OUTPUT_DATA_PATH <- setup_ctx$OUTPUT_DATA_PATH
OUTPUT_PLOTS_PATH <- setup_ctx$OUTPUT_PLOTS_PATH
INTERMEDIATE_RESULTS_PATH <- setup_ctx$INTERMEDIATE_RESULTS_PATH

In [ ]:
# SNT config

# Load config file
CONFIG_FILE_NAME <- "SNT_config.json"
config_json <- tryCatch({ fromJSON(file.path(CONFIG_PATH, CONFIG_FILE_NAME)) },
    error = function(e) {
        msg <- paste0("Erreur à charger le fichier de configuration", conditionMessage(e))  
        cat(msg)   
        stop(msg) 
    })

msg <- paste0("Configuration SNT chargée depuis le fichier : ", file.path(CONFIG_PATH, CONFIG_FILE_NAME)) 
pipeline_msg(msg)

# Set config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
seasonality_dataset <- config_json$SNT_DATASET_IDENTIFIERS$SNT_SEASONALITY_RAINFALL

print(paste("Code pays : ", COUNTRY_CODE))

pipeline_msg("Importation des fonctions, librairies et dépendances achevée.")

### 1.1 Paramètres

Importation des paramètres enregistrés lors de la dernière exécution des calculs.

In [ ]:
pipeline_msg("Importation des paramètres en cours.")

parameters_file <- paste0(COUNTRY_CODE, "_parameters.json")

# Try to load parameters from dataset, if it exists
parameters <- tryCatch({
    get_latest_dataset_file_in_memory(seasonality_dataset, parameters_file)
}, error = function(e) {
    pipeline_msg(paste0("[AVERTISSEMENT] Les paramètres n'ont pas pu être chargés depuis le dataset, les valeurs par défaut seront utilisées : ", conditionMessage(e)), level = "warning")
    NULL
})

In [ ]:
# Default values (to be used if missing in the parameters file)
minimum_month_block_size <- as.integer(3)
maximum_month_block_size <- as.integer(5)
threshold_for_seasonality <- 0.6
threshold_proportion_seasonal_years <- 0.5
use_calendar_year_denominator <- FALSE

In [ ]:
# Override defaults if parameter list exists and the respective params also exist and are valid
minimum_month_block_size <- get_param(parameters, "MINIMUM_MONTH_BLOCK_SIZE", minimum_month_block_size, as.integer)
maximum_month_block_size <- get_param(parameters, "MAXIMUM_MONTH_BLOCK_SIZE", maximum_month_block_size, as.integer)
threshold_for_seasonality <- get_param(parameters, "THRESHOLD_FOR_SEASONALITY", threshold_for_seasonality, as.numeric)
threshold_proportion_seasonal_years <- get_param(parameters, "THRESHOLD_PROPORTION_SEASONAL_YEARS", threshold_proportion_seasonal_years, as.numeric)
use_calendar_year_denominator <- get_param(parameters, "USE_CALENDAR_YEAR_DENOMINATOR", USE_CALENDAR_YEAR_DENOMINATOR, as.logical)

# Logging what values for the params are used
if (!is.null(parameters) && is.list(parameters)) {
  pipeline_msg(paste0("Paramètres chargés (valeurs par défaut si manquants) : ", parameters_file))
  pipeline_msg(paste(names(parameters), ": ", parameters))
} else {
  pipeline_msg("Paramètres manquants, valeurs par défaut utilisées")
}

### 1.2 Variables dérivées pour les titres et noms de fichiers

In [ ]:
# Create denominator suffix for filenames based on denominator method
if (isTRUE(use_calendar_year_denominator)) {
    denominator_type <- "calendar"
} else {
    denominator_type <- "sliding"
}

pipeline_msg(paste("Type de dénominateur :", denominator_type))


In [ ]:
# Global variables
type_of_seasonality <- "rainfall"
fr_type_of_seasonality <- "pluviométrie"
data_source <- 'ERA5'

# Space and time columns
admin_level <- 'ADM2'
admin_upper <- 'ADM1'
admin_id_col <- paste(admin_level, toupper('id'), sep = '_')
admin_name_col <- paste(admin_level, toupper('name'), sep = '_')
year_col <- 'YEAR'
month_col <- 'MONTH'
period_cols <- c(year_col, month_col)

# Formatted percentages for plot labels
formatted_threshold_for_seasonality <- sprintf("%d%%", round(threshold_for_seasonality * 100))
formatted_threshold_proportion_seasonal_years <- sprintf("%d%%", round(threshold_proportion_seasonal_years * 100))

# Subtitle text for plots (shows parameter values)
subtitle_text <- paste0(
    "Proportion minimale de pluviométrie annuelle : ", formatted_threshold_for_seasonality, 
    "\nProportion minimale d'années avec schéma saisonnier : ", formatted_threshold_proportion_seasonal_years,
    "\nDurée de la saison pluvieuse : ", minimum_month_block_size, "-", maximum_month_block_size, " mois",
    "\nType de dénominateur : ", denominator_type
)

# Percentage particles for plot filenames
particle_threshold_for_seasonality <- glue::glue("{round(threshold_for_seasonality * 100)}PCT")
particle_threshold_proportion_seasonal_years <- glue::glue("{round(threshold_proportion_seasonal_years * 100)}PCT_YRS")
prefix_for_filenames <- glue::glue("{COUNTRY_CODE}_{data_source}_{admin_level}_{particle_threshold_for_seasonality}_{type_of_seasonality}_{particle_threshold_proportion_seasonal_years}_{denominator_type}")

pipeline_msg("Importation des paramètres achevée.")

## 2. Chargement et pré-processing des données à cartographier

### 2.1 Chargement des données

**Les données utilisées**

Toutes les données utilisées dans ce rapport sont agrégées au niveau administratif ADM2:
* Données spatiales : fond de carte.
* Données de pluviométrie : données sur les précipitations mensuelles.
* Données de saisonnalité : les résultats calculés par le notebook de calcul de la saisonnalité basée sur les précipitations.

In [ ]:
pipeline_msg("Importation et préparation des données.")

# Load spatial file from dataset
spatial_data_filename <- paste(COUNTRY_CODE, "shapes.geojson", sep = "_")

spatial_data <- tryCatch({ get_latest_dataset_file_in_memory(dhis2_dataset, spatial_data_filename) }, 
                  error = function(e) {
                      msg <- paste("Error while loading DHIS2 Shapes data for: " , COUNTRY_CODE, conditionMessage(e))
                      cat(msg)
                      stop(msg)
                      })

In [ ]:
# import rainfall and seasonality data

rainfall_data <- read_parquet(file.path(OUTPUT_DATA_PATH, glue("{COUNTRY_CODE}_rainfall_row_seasonality.parquet")))
setDT(rainfall_data)

seasonality_data <- read_parquet(file.path(OUTPUT_DATA_PATH, glue("{COUNTRY_CODE}_rainfall_seasonality.parquet")))
setDT(seasonality_data)

### 2.2 Préparation aux visualisations

#### 2.2.1 Données de pluviométrie

In [ ]:
# create the administrative dataset
admin_data <- setDT(st_drop_geometry(spatial_data))

# keep only useful columns in rainfall data
rainfall_data <- rainfall_data[, .SD, .SDcols = c("ADM2_ID", "YEAR", "MONTH", "MEAN_EST")]

# keep only full years
rainfall_data <- filter_complete_groups(input_dt = rainfall_data, upper_colname = "YEAR", lower_colname = "MONTH")

# join with amin data
rainfall_data <- merge.data.table(admin_data, rainfall_data, by = "ADM2_ID")

# sum by year/month and admin unit
rainfall_data <- rainfall_data[
  ,
  lapply(.SD, sum, na.rm = TRUE),
  by = c("ADM1_NAME", "YEAR", "MONTH"),
  .SDcols = c("MEAN_EST")
]

rainfall_data <- add_year_month(
  input_dt = rainfall_data,
  year_colname = year_col,
  month_colname = month_col
)

#### 2.2.1 Données de saisonnalité

In [ ]:
# merge polygon file with seasonality dataset
plot_data <- merge(spatial_data, seasonality_data, by = c('ADM1_NAME','ADM1_ID', admin_id_col, admin_name_col), all = TRUE)

pipeline_msg("Importation et préparation des données achevée.")

## 3. Génération des cartes de résultat

### Patterns de pluviométrie

Ce graphique montre **les schémas de précipitations mensuelles** au niveau administratif supérieur, soit **au niveau ADM1**.

In [ ]:
pipeline_msg("Création de contenu: visualisation et texte.")

# make text
min_year <- rainfall_data[get(year_col) == min(get(year_col)), unique(get(year_col))]
max_year <- rainfall_data[get(year_col) == max(get(year_col)), unique(get(year_col))]

rainfall_intro_text <- glue("Sur l'ensemble de la période analysée, de {min_year} à {max_year}")
top_admin_upper <- get_top_summed_group(rainfall_data, "MEAN_EST", "ADM1_NAME")[[1]]

# different behavior if one vs several upper-level admin units have the same top value
if (get_top_summed_group(rainfall_data, "MEAN_EST", "ADM1_NAME")[[3]] == 1){
  top_admin_upper <- to_title_case(top_admin_upper)

  rainfall_max_text <- glue("l'unité administrative {admin_upper} ayant enregistré les valeurs les plus élevées de précipitations est {top_admin_upper}.")
} else{
  top_admin_upper <- paste(top_admin_upper, collapse =", ")
  top_admin_upper <- to_title_case(paste(top_admin_upper, collapse =", "))

  rainfall_max_text <- glue("les unités {admin_upper} ayant enregistré les valeurs les plus élevées de précipitations sont {top_admin_upper}.")
}

rainfall_text <- paste(rainfall_intro_text, rainfall_max_text, collapse = ", ")


In [ ]:
print(rainfall_text)

In [ ]:
precipitation_ridge_plot <- make_ridgeline_plot(
  dt=rainfall_data,
  x_colname = "year_month",
  y_colname = "ADM1_NAME",
  height_colname = "MEAN_EST",
  year_colname = year_col,
  month_colname = month_col,
  plot_title = "Précipitations mensuelles",
  plot_subtitle = glue("Niveau {admin_upper}"),
  plot_caption = glue("Données: {data_source}"),
  scale_constant = 2
)


In [ ]:
precipitation_ridge_plot

In [ ]:
# Save the plot
plot_filename <- glue("{COUNTRY_CODE}_{data_source}_rainfall.png")

ggsave(
  filename=file.path(OUTPUT_PLOTS_PATH, plot_filename),
  plot=precipitation_ridge_plot,
  bg="white",
  dpi = 300
)

### Présence de saisonnalité

Cette carte montre **la classification des unités administratives (saisonnière vs non saisonnière)** selon les seuils configurés dans cette instance du pipeline.

In [ ]:
n_units <- nrow(seasonality_data)
n_seasonal_units <- sum(seasonality_data$SEASONALITY_RAINFALL, na.rm = TRUE)
condition_seasonality <- n_seasonal_units > 0

text_seasonality <- glue("Sur l'ensemble des {n_units} unités administratives {admin_level}, {n_seasonal_units} ont été classées comme saisonnières, selon les critères et seuils définis.")


In [ ]:
print(text_seasonality)

In [ ]:
# Seasonality classification plot (seasonal vs non-seasonal)
seasonality_plot <- make_seasonality_plot(
  spatial_seasonality_df=plot_data,
  seasonality_colname=paste('SEASONALITY', toupper(type_of_seasonality), sep = "_"),
  plot_title=paste("Saisonnalité:", fr_type_of_seasonality, sep = ' '),
  plot_subtitle = subtitle_text,
  legend_title = NULL,
  plot_caption = glue("Données: {data_source}"),
  seasonal_color = "#FFDAB9",
  seasonal_label = "Saisonnier",
  not_seasonal_color = "#B3E0FF",
  not_seasonal_label = "Non saisonnier"
)


In [ ]:
seasonality_plot

In [ ]:
# Save
filename_seasonality_plot <- glue::glue("{prefix_for_filenames}_seasonality_map.png")
ggsave(
  filename=file.path(OUTPUT_PLOTS_PATH, filename_seasonality_plot),
  plot=seasonality_plot,
  bg="white",
  dpi = 300
)

In [ ]:
# If there is at least one seasonal ADM2 unit, display formatted markdown

if (condition_seasonality) {
    IRdisplay::display_markdown("### Début du bloc saisonnier")
}


In [ ]:
# If there is at least one seasonal ADM2 unit, display formatted markdown

if (condition_seasonality) {
    IRdisplay::display_markdown("Cette carte montre **le mois de démarrage du bloc saisonnier**, pour les unités administratives classées comme saisonnières en termes de précipitations.")
}

In [ ]:
# If there is at least one seasonal ADM2 unit
if (condition_seasonality) {
  
    # Beginning month of seasonal block plot
    season_start_month_col <- paste('SEASONAL_BLOCK_START_MONTH', toupper(type_of_seasonality), sep = "_")

    start_month_plot <- make_season_start_month_plot(
        plot_data = plot_data,
        season_start_month_col = "SEASONAL_BLOCK_START_MONTH_RAINFALL",
        color_vector = make_month_colors(),
        color_labels = make_month_labels_fr(),
        plot_title = "Début de la saison pluvieuse",
        plot_subtitle = subtitle_text,
        plot_caption = glue("Données: {data_source}"),
        legend_title = "Mois de début",
        missing_label = "Non saisonnier"
    )
  
    print(start_month_plot)

    # save the plot
    filename_start_month_plot <- glue::glue("{prefix_for_filenames}_season_start_map.png")
    ggsave(
    filename = file.path(OUTPUT_PLOTS_PATH, filename_start_month_plot),
    plot = start_month_plot,
    bg = "white",
    dpi = 300,
    width = 12,
    height = 8
    )
  
}
  

In [ ]:
# If there is at least one seasonal ADM2 unit, display formatted markdown

if (condition_seasonality) {
    IRdisplay::display_markdown("### Durée de la saison pluvieuse")
}


In [ ]:
# If there is at least one seasonal ADM2 unit, display formatted markdown

if (condition_seasonality) {
    IRdisplay::display_markdown("La carte montre **la durée (en mois) de la saison pluvieuse par unité administrative**.")
}

In [ ]:
# If there is at least one seasonal ADM2 unit
if (condition_seasonality) {

  # Duration of seasonality plot
  duration_plot <- make_season_duration_plot(
    spatial_seasonality_df=plot_data,
    seasonality_duration_colname=paste('SEASONAL_BLOCK_DURATION', toupper(type_of_seasonality), sep = "_"),
    plot_title="Durée de la saison pluvieuse (mois)",
    plot_subtitle = subtitle_text,
    plot_caption = glue("Données: {data_source}"),
    legend_title = "Durée (mois)",
    color_vector = c("#FDDECE", "#F07A58", "#A8381E"),
    none_label="Pas saisonnier"
  )

  print(duration_plot)

  # Save with suffix
  filename_duration_plot <- glue::glue("{prefix_for_filenames}_season_duration_map.png")
  ggsave(
    filename=file.path(OUTPUT_PLOTS_PATH, filename_duration_plot),
    plot=duration_plot,
    bg = "white", dpi = 300, width = 12, height = 8
  )

}

In [ ]:
# If there is at least one seasonal ADM2 unit, display formatted markdown

if (condition_seasonality) {
    IRdisplay::display_markdown("### Proportion de pluie durant la saison pluvieuse")
}

In [ ]:
# If there is at least one seasonal ADM2 unit, display formatted markdown

if (condition_seasonality) {
    IRdisplay::display_markdown("La carte suivante montre **la part de pluie annuelle qui a lieu durant la saison pluvieuse**, par unité administrative.")
}

In [ ]:
# If there is at least one seasonal ADM2 unit
if (condition_seasonality) {
  
  # Make text on what proportion of rain falls during the seasonal block
  rain_proportion_hi <- round(max(seasonality_data$RAIN_PROPORTION, na.rm = TRUE)*100)
  rain_proportion_mean <- round(mean(seasonality_data$RAIN_PROPORTION, na.rm = TRUE)*100)

  rain_proportion_intro_text <- glue("Dans les zones où l'on remarque un schéma pluviométrique saisonnier, en moyenne {rain_proportion_mean}% des précipitations ont lieu durant la saison pluvieuse.")


  top_admin_units <- seasonality_data[RAIN_PROPORTION == max(seasonality_data$RAIN_PROPORTION, na.rm = TRUE), unique(get(admin_name_col))]
  # different behavior if one vs several admin units  have the same top value
  if (length(top_admin_units) == 1){
    top_admin_units <- to_title_case(top_admin_units)

    seasonality_max_text <- glue("{top_admin_units}")
  } else{
    top_admin_units <- paste(top_admin_units, collapse =", ")
    top_admin_units <- to_title_case(paste(top_admin_units, collapse =", "))
  }

  rain_proportion_max_text <- glue("La part la plus élevée, de {rain_proportion_hi}%, est enregistrée en {top_admin_units}.")

  rain_proportion_text <- paste(rain_proportion_intro_text, rain_proportion_max_text, collapse = ". ")

  print(rain_proportion_text)

}

In [ ]:
# If there is at least one seasonal ADM2 unit
if (condition_seasonality) {
  
  # Map: Proportion of annual rainfall in seasonal block
  proportion_plot <- make_season_proportion_plot(
      plot_data = plot_data,
      proportion_colname = "RAIN_PROPORTION",
      plot_title = "Précipitations durant la saison pluvieuse (%)",
      plot_subtitle = subtitle_text,
      plot_caption = glue("Données: {data_source}"),
      legend_title = NULL,
      color_vector = c(
        "#C8DDD9",
        "#9DBFBB",
        "#5E9490",
        "#2E6460",
        "#264A48"
        )
      )

  if (!is.null(proportion_plot)) {
    print(proportion_plot)

    # Save with suffix
    filename_proportion_plot <- glue::glue("{prefix_for_filenames}_season_pct_map.png")
    ggsave(
      filename=file.path(OUTPUT_PLOTS_PATH, filename_proportion_plot),
      plot=proportion_plot,
      bg="white", dpi=300, width=10, height=8
    )
  } else {
    cat('RAIN_PROPORTION column not found in data. Run the updated code notebook first.\n')
  }

}


In [ ]:

pipeline_msg("Création de contenu achevée.")